In [3]:
import pandas as pd 
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv("/home/sjoon/projects/brain_connectivity_classifier/results/experiments/Exp_LR_RM_bestparam/predictions/cv_validation_predictions.csv")
print(df.head())
print(df.shape)

    subject_id  true_region  predicted_region
0  sub-0002_P2            0                 0
1  sub-0002_P2            1                 1
2  sub-0002_P2            2                 2
3  sub-0002_P2            3                 3
4  sub-0002_P2            4                 4
(51968, 3)


In [10]:
region_list = pd.read_csv("/home/sjoon/projects/brain_connectivity_classifier/data/processed/region_list.csv")
region_list.head()

,region
0,LH_VisCent_ExStr_2
1,LH_VisCent_ExStr_1
2,LH_VisCent_Striate_1
3,LH_VisCent_ExStr_3
4,LH_VisCent_ExStr_4


In [11]:
region_list = region_list['region'].tolist()
print(region_list[0:5])

['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1', 'LH_VisCent_ExStr_3', 'LH_VisCent_ExStr_4']


In [12]:
# copy
df = df.copy()

# Now add the region names (warning will disappear)
region_mapping = {i: name for i, name in enumerate(region_list)}
df['true_region_name'] = df['true_region'].map(region_mapping)
df['predicted_region_name'] = df['predicted_region'].map(region_mapping)

# Drop subject_id from error_samples
print(df.head())


    subject_id  true_region  predicted_region      true_region_name  \
0  sub-0002_P2            0                 0    LH_VisCent_ExStr_2   
1  sub-0002_P2            1                 1    LH_VisCent_ExStr_1   
2  sub-0002_P2            2                 2  LH_VisCent_Striate_1   
3  sub-0002_P2            3                 3    LH_VisCent_ExStr_3   
4  sub-0002_P2            4                 4    LH_VisCent_ExStr_4   

  predicted_region_name  
0    LH_VisCent_ExStr_2  
1    LH_VisCent_ExStr_1  
2  LH_VisCent_Striate_1  
3    LH_VisCent_ExStr_3  
4    LH_VisCent_ExStr_4  


In [13]:
# No of samples incorrectly predicted
error_samples = df[df['true_region'] != df['predicted_region']]
error_samples.shape[0]

4407

In [14]:
# calculate accuracy
y_true = df['true_region']
y_pred = df['predicted_region']
accuracy = accuracy_score(y_true, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9151978140394089


In [15]:

print(error_samples.shape)
error_samples.head()

(4407, 5)


,subject_id,true_region,predicted_region,true_region_name,predicted_region_name
13,sub-0002_P2,13,122,LH_SomMotA_2,RH_SomMotA_11
15,sub-0002_P2,15,92,LH_SomMotA_4,LH_DefaultB_PFCv_2
28,sub-0002_P2,28,0,LH_DorsAttnA_TempOcc_1,LH_VisCent_ExStr_2
99,sub-0002_P2,99,196,LH_TempPar_2,RH_TempPar_1
102,sub-0002_P2,102,4,RH_VisCent_Striate_1,LH_VisCent_ExStr_4


In [16]:
error_samples.isnull().sum()

subject_id               0
true_region              0
predicted_region         0
true_region_name         0
predicted_region_name    0
dtype: int64

In [22]:
# Comprehensive detailed error categorization (must sum to 100%)
print("="*60)
print("DETAILED ERROR CATEGORIZATION (Total = 100%)")
print("="*60)
print(f"Total errors: {len(error_samples)}\n")

# Copy to avoid warnings
error_samples = error_samples.copy()

# Helper function to extract hemisphere (handles both cortical and subcortical)
def get_hemisphere(region_name):
    if region_name.startswith('LH_'):
        return 'LH'
    elif region_name.startswith('RH_'):
        return 'RH'
    elif region_name.endswith('-lh'):
        return 'LH'
    elif region_name.endswith('-rh'):
        return 'RH'
    else:
        return 'Unknown'

# Helper function to extract region base (without hemisphere prefix/suffix)
def get_region_base(region_name):
    if region_name.startswith('LH_') or region_name.startswith('RH_'):
        return region_name[3:]  # Remove 'LH_' or 'RH_'
    elif region_name.endswith('-lh') or region_name.endswith('-rh'):
        return region_name[:-3]  # Remove '-lh' or '-rh'
    else:
        return region_name

# Helper function to extract network (cortical only)
def get_network(region_name):
    if region_name.startswith(('LH_', 'RH_')):
        parts = region_name.split('_')
        return parts[1] if len(parts) > 1 else 'Unknown'
    else:
        # Subcortical - no network structure
        return 'Subcortical'

# Extract hemisphere and network information
error_samples['true_hemisphere'] = error_samples['true_region_name'].apply(get_hemisphere)
error_samples['predicted_hemisphere'] = error_samples['predicted_region_name'].apply(get_hemisphere)
error_samples['true_region_base'] = error_samples['true_region_name'].apply(get_region_base)
error_samples['predicted_region_base'] = error_samples['predicted_region_name'].apply(get_region_base)
error_samples['true_network'] = error_samples['true_region_name'].apply(get_network)
error_samples['predicted_network'] = error_samples['predicted_region_name'].apply(get_network)

# Extract anatomical location (third component, cortical only)
# DIAGNOSTIC: Region and Network Distribution
print("="*60)
print("REGION & NETWORK DISTRIBUTION")
print("="*60)

# Count regions by hemisphere
lh_count = error_samples['true_hemisphere'].value_counts().get('LH', 0) + error_samples['predicted_hemisphere'].value_counts().get('LH', 0)
rh_count = error_samples['true_hemisphere'].value_counts().get('RH', 0) + error_samples['predicted_hemisphere'].value_counts().get('RH', 0)

# Get unique networks
unique_networks = sorted(set(error_samples['true_network'].unique()) | set(error_samples['predicted_network'].unique()))

print(f"Total regions in dataset: {len(region_list)}")
print(f"  Left hemisphere (LH): {sum(1 for r in region_list if get_hemisphere(r) == 'LH')}")
print(f"  Right hemisphere (RH): {sum(1 for r in region_list if get_hemisphere(r) == 'RH')}")
print(f"\nNetworks involved in errors:")
for network in unique_networks:
    count_true = (error_samples['true_network'] == network).sum()
    count_pred = (error_samples['predicted_network'] == network).sum()
    print(f"  {network}: {max(count_true, count_pred)} errors")
print()

error_samples['true_location'] = error_samples['true_region_name'].str.split('_').str[2]
error_samples['predicted_location'] = error_samples['predicted_region_name'].str.split('_').str[2]

# Extract subregion number (last component)
error_samples['true_subregion'] = error_samples['true_region_name'].str.split('_').str[-1]
error_samples['predicted_subregion'] = error_samples['predicted_region_name'].str.split('_').str[-1]

print("="*60)
print("CATEGORY 1: SYMMETRIC HEMISPHERE CONFUSIONS")
print("="*60)
# Category 1: Symmetric hemisphere confusions (LH ↔ RH, same region)
symmetric_hemi_mask = ((error_samples['true_hemisphere'] != error_samples['predicted_hemisphere']) & 
                       (error_samples['true_region_base'] == error_samples['predicted_region_base']))
symmetric_hemi = symmetric_hemi_mask.sum()

# Breakdown: LH→RH vs RH→LH
lh_to_rh = ((error_samples['true_hemisphere'] == 'LH') & 
            (error_samples['predicted_hemisphere'] == 'RH') & 
            (error_samples['true_region_base'] == error_samples['predicted_region_base'])).sum()
rh_to_lh = ((error_samples['true_hemisphere'] == 'RH') & 
            (error_samples['predicted_hemisphere'] == 'LH') & 
            (error_samples['true_region_base'] == error_samples['predicted_region_base'])).sum()

print(f"Total symmetric hemisphere confusions: {symmetric_hemi} ({symmetric_hemi/len(error_samples)*100:.1f}%)")
print(f"  - LH → RH: {lh_to_rh} ({lh_to_rh/len(error_samples)*100:.1f}%)")
print(f"  - RH → LH: {rh_to_lh} ({rh_to_lh/len(error_samples)*100:.1f}%)")

# Most confused symmetric pairs
if symmetric_hemi > 0:
    symmetric_pairs = error_samples[symmetric_hemi_mask].groupby('true_region_base').size().sort_values(ascending=False)
    print(f"\n  Top 5 most confused symmetric regions:")
    for region, count in symmetric_pairs.head(5).items():
        print(f"    {region}: {count} errors")

print("\n" + "="*60)
print("CATEGORY 2: SAME HEMISPHERE, SAME NETWORK")
print("="*60)
same_hemi_same_net_mask = ((error_samples['true_hemisphere'] == error_samples['predicted_hemisphere']) & 
                           (error_samples['true_network'] == error_samples['predicted_network']))
same_hemi_same_net = same_hemi_same_net_mask.sum()

print(f"Total same hemisphere, same network: {same_hemi_same_net} ({same_hemi_same_net/len(error_samples)*100:.1f}%)")

# Breakdown by hemisphere
lh_same_net = ((error_samples['true_hemisphere'] == 'LH') & 
               (error_samples['predicted_hemisphere'] == 'LH') & 
               (error_samples['true_network'] == error_samples['predicted_network'])).sum()
rh_same_net = ((error_samples['true_hemisphere'] == 'RH') & 
               (error_samples['predicted_hemisphere'] == 'RH') & 
               (error_samples['true_network'] == error_samples['predicted_network'])).sum()

print(f"  - Left hemisphere: {lh_same_net} ({lh_same_net/len(error_samples)*100:.1f}%)")
print(f"  - Right hemisphere: {rh_same_net} ({rh_same_net/len(error_samples)*100:.1f}%)")

# Breakdown by network
if same_hemi_same_net > 0:
    network_breakdown = error_samples[same_hemi_same_net_mask].groupby('true_network').size().sort_values(ascending=False)
    print(f"\n  Errors by network:")
    for network, count in network_breakdown.items():
        print(f"    {network}: {count} errors ({count/len(error_samples)*100:.1f}%)")
    
    # Adjacent subregion confusions (e.g., region_1 ↔ region_2)
    adjacent_subregion = ((error_samples['true_hemisphere'] == error_samples['predicted_hemisphere']) & 
                         (error_samples['true_network'] == error_samples['predicted_network']) &
                         (error_samples['true_location'] == error_samples['predicted_location'])).sum()
    print(f"\n  - Adjacent subregions (same location): {adjacent_subregion} ({adjacent_subregion/len(error_samples)*100:.1f}%)")
    print(f"  - Different locations (same network): {same_hemi_same_net - adjacent_subregion} ({(same_hemi_same_net - adjacent_subregion)/len(error_samples)*100:.1f}%)")

print("\n" + "="*60)
print("CATEGORY 3: SAME HEMISPHERE, DIFFERENT NETWORK")
print("="*60)
same_hemi_diff_net_mask = ((error_samples['true_hemisphere'] == error_samples['predicted_hemisphere']) & 
                           (error_samples['true_network'] != error_samples['predicted_network']))
same_hemi_diff_net = same_hemi_diff_net_mask.sum()

print(f"Total same hemisphere, different network: {same_hemi_diff_net} ({same_hemi_diff_net/len(error_samples)*100:.1f}%)")

# Breakdown by hemisphere
lh_diff_net = ((error_samples['true_hemisphere'] == 'LH') & 
               (error_samples['predicted_hemisphere'] == 'LH') & 
               (error_samples['true_network'] != error_samples['predicted_network'])).sum()
rh_diff_net = ((error_samples['true_hemisphere'] == 'RH') & 
               (error_samples['predicted_hemisphere'] == 'RH') & 
               (error_samples['true_network'] != error_samples['predicted_network'])).sum()

print(f"  - Left hemisphere: {lh_diff_net} ({lh_diff_net/len(error_samples)*100:.1f}%)")
print(f"  - Right hemisphere: {rh_diff_net} ({rh_diff_net/len(error_samples)*100:.1f}%)")

# Most common network confusions
if same_hemi_diff_net > 0:
    network_confusions = error_samples[same_hemi_diff_net_mask].groupby(['true_network', 'predicted_network']).size().sort_values(ascending=False)
    print(f"\n  Top 10 network confusion pairs:")
    for (true_net, pred_net), count in network_confusions.head(10).items():
        print(f"    {true_net} → {pred_net}: {count} errors ({count/len(error_samples)*100:.1f}%)")

print("\n" + "="*60)
print("CATEGORY 4: DIFFERENT HEMISPHERE, DIFFERENT REGION")
print("="*60)
diff_hemi_nonsym_mask = ((error_samples['true_hemisphere'] != error_samples['predicted_hemisphere']) & 
                         (error_samples['true_region_base'] != error_samples['predicted_region_base']))
diff_hemi_nonsym = diff_hemi_nonsym_mask.sum()

print(f"Total different hemisphere, different region: {diff_hemi_nonsym} ({diff_hemi_nonsym/len(error_samples)*100:.1f}%)")

# Breakdown: LH→RH vs RH→LH
lh_to_rh_nonsym = ((error_samples['true_hemisphere'] == 'LH') & 
                   (error_samples['predicted_hemisphere'] == 'RH') & 
                   (error_samples['true_region_base'] != error_samples['predicted_region_base'])).sum()
rh_to_lh_nonsym = ((error_samples['true_hemisphere'] == 'RH') & 
                   (error_samples['predicted_hemisphere'] == 'LH') & 
                   (error_samples['true_region_base'] != error_samples['predicted_region_base'])).sum()

print(f"  - LH → RH (different region): {lh_to_rh_nonsym} ({lh_to_rh_nonsym/len(error_samples)*100:.1f}%)")
print(f"  - RH → LH (different region): {rh_to_lh_nonsym} ({rh_to_lh_nonsym/len(error_samples)*100:.1f}%)")

# Same network but different hemisphere?
diff_hemi_same_net = ((error_samples['true_hemisphere'] != error_samples['predicted_hemisphere']) & 
                      (error_samples['true_region_base'] != error_samples['predicted_region_base']) &
                      (error_samples['true_network'] == error_samples['predicted_network'])).sum()
diff_hemi_diff_net = ((error_samples['true_hemisphere'] != error_samples['predicted_hemisphere']) & 
                      (error_samples['true_region_base'] != error_samples['predicted_region_base']) &
                      (error_samples['true_network'] != error_samples['predicted_network'])).sum()

print(f"\n  - Same network (homologous): {diff_hemi_same_net} ({diff_hemi_same_net/len(error_samples)*100:.1f}%)")
print(f"  - Different network: {diff_hemi_diff_net} ({diff_hemi_diff_net/len(error_samples)*100:.1f}%)")

# Most common cross-hemisphere confusions
if diff_hemi_nonsym > 0:
    cross_hemi_pairs = error_samples[diff_hemi_nonsym_mask].groupby(['true_region_name', 'predicted_region_name']).size().sort_values(ascending=False)
    print(f"\n  Top 5 cross-hemisphere confusion pairs:")
    for (true_reg, pred_reg), count in cross_hemi_pairs.head(5).items():
        print(f"    {true_reg} → {pred_reg}: {count} errors")

# Verify total
total = symmetric_hemi + same_hemi_same_net + same_hemi_diff_net + diff_hemi_nonsym

print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)

# Create detailed summary
categories_detailed = {
    '1. Symmetric hemisphere (LH↔RH, same region)': symmetric_hemi,
    '   1a. LH → RH': lh_to_rh,
    '   1b. RH → LH': rh_to_lh,
    '2. Same hemisphere, same network': same_hemi_same_net,
    '   2a. Left hemisphere': lh_same_net,
    '   2b. Right hemisphere': rh_same_net,
    '3. Same hemisphere, different network': same_hemi_diff_net,
    '   3a. Left hemisphere': lh_diff_net,
    '   3b. Right hemisphere': rh_diff_net,
    '4. Different hemisphere, different region': diff_hemi_nonsym,
    '   4a. LH → RH (non-symmetric)': lh_to_rh_nonsym,
    '   4b. RH → LH (non-symmetric)': rh_to_lh_nonsym,
}

for category, count in categories_detailed.items():
    percentage = (count / len(error_samples)) * 100
    print(f"{category:50s}: {count:5d} ({percentage:5.1f}%)")

print(f"\n{'TOTAL':50s}: {total:5d} ({(total/len(error_samples))*100:5.1f}%)")
print(f"{'Verification (should match total errors)':50s}: {len(error_samples):5d}")

# High-level summary
print("\n" + "="*60)
print("HIGH-LEVEL INSIGHTS")
print("="*60)
total_hemisphere_errors = symmetric_hemi + diff_hemi_nonsym
total_same_hemisphere = same_hemi_same_net + same_hemi_diff_net

print(f"Hemisphere-related errors: {total_hemisphere_errors} ({total_hemisphere_errors/len(error_samples)*100:.1f}%)")
print(f"  - Systematic (symmetric): {symmetric_hemi} ({symmetric_hemi/total_hemisphere_errors*100:.1f}% of hemisphere errors)")
print(f"  - Non-systematic: {diff_hemi_nonsym} ({diff_hemi_nonsym/total_hemisphere_errors*100:.1f}% of hemisphere errors)")
print(f"\nSame-hemisphere errors: {total_same_hemisphere} ({total_same_hemisphere/len(error_samples)*100:.1f}%)")
print(f"  - Within network: {same_hemi_same_net} ({same_hemi_same_net/total_same_hemisphere*100:.1f}% of same-hemisphere errors)")
print(f"  - Cross network: {same_hemi_diff_net} ({same_hemi_diff_net/total_same_hemisphere*100:.1f}% of same-hemisphere errors)")

DETAILED ERROR CATEGORIZATION (Total = 100%)
Total errors: 4407

REGION & NETWORK DISTRIBUTION
Total regions in dataset: 232
  Left hemisphere (LH): 116
  Right hemisphere (RH): 116

Networks involved in errors:
  ContA: 297 errors
  ContB: 255 errors
  ContC: 28 errors
  DefaultA: 224 errors
  DefaultB: 290 errors
  DefaultC: 64 errors
  DorsAttnA: 211 errors
  DorsAttnB: 152 errors
  LimbicA: 378 errors
  LimbicB: 139 errors
  SalVentAttnA: 296 errors
  SalVentAttnB: 188 errors
  SomMotA: 216 errors
  SomMotB: 167 errors
  Subcortical: 1345 errors
  TempPar: 176 errors
  VisCent: 93 errors
  VisPeri: 107 errors

CATEGORY 1: SYMMETRIC HEMISPHERE CONFUSIONS
Total symmetric hemisphere confusions: 73 (1.7%)
  - LH → RH: 31 (0.7%)
  - RH → LH: 42 (1.0%)

  Top 5 most confused symmetric regions:
    aGP: 16 errors
    pGP: 14 errors
    LimbicA_TempPole_3: 7 errors
    lAMY: 6 errors
    TempPar_2: 3 errors

CATEGORY 2: SAME HEMISPHERE, SAME NETWORK
Total same hemisphere, same network: 491